In [87]:
import pandas as pd
import numpy as np

# Separate consumption data and label
usage = df.drop(columns=["FLAG"]).apply(pd.to_numeric, errors="coerce")
usage = usage.fillna(usage.median())

features = pd.DataFrame()

# Basic consumption behaviour
features["mean_usage"] = usage.mean(axis=1)
features["median_usage"] = usage.median(axis=1)
features["max_usage"] = usage.max(axis=1)
features["min_usage"] = usage.min(axis=1)
features["std_usage"] = usage.std(axis=1)

# Difference between highest and lowest consumption
features["usage_range"] = (
    features["max_usage"] - features["min_usage"]
)

# Variation in consumption
features["variation"] = (
    features["std_usage"] /
    (features["mean_usage"] + 1)
)

# Number of zero / near-zero consumption days
features["zero_days"] = (usage <= 1).sum(axis=1)

# Day-to-day changes
changes = usage.diff(axis=1)

features["avg_change"] = changes.abs().mean(axis=1)
features["max_drop"] = changes.min(axis=1).abs()
features["max_spike"] = changes.max(axis=1)

# Compare first half and second half
half = usage.shape[1] // 2

first_half = usage.iloc[:, :half].mean(axis=1)
second_half = usage.iloc[:, half:].mean(axis=1)

features["usage_change_ratio"] = (
    second_half / (first_half + 1)
)

# Label
features["FLAG"] = df["FLAG"].values

print("New WattGuard dataset:", features.shape)

features.head()

KeyError: "['FLAG'] not found in axis"

In [ ]:
features.groupby("FLAG").mean().round(2)

,mean_usage,median_usage,max_usage,min_usage,std_usage,usage_range,variation,zero_days,avg_change,max_drop,max_spike,usage_change_ratio
FLAG,,,,,,,,,,,,
0,6.12,6.01,11.19,2.07,2.43,9.12,0.33,9.66,1.90,6.33,6.52,0.75
1,28.59,16.84,322.76,6.37,64.26,316.39,0.42,7.98,27.95,306.99,307.56,19.43


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    fbeta_score
)
import numpy as np
import pandas as pd

# Separate features and label
X = features.drop(columns=["FLAG"])
y = features["FLAG"]

# 80% development + 20% final test
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split development data again:
# 60% train + 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

# WattGuard V2 model
model_v2 = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model_v2.fit(X_train, y_train)

# Validation probabilities
val_prob = model_v2.predict_proba(X_val)[:, 1]

# Find a threshold that gives more importance to theft recall
best_threshold = 0.50
best_score = 0

for threshold in np.arange(0.10, 0.71, 0.02):
    val_pred = (val_prob >= threshold).astype(int)

    score = fbeta_score(
        y_val,
        val_pred,
        beta=2
    )

    if score > best_score:
        best_score = score
        best_threshold = threshold

print("Best threshold:", round(best_threshold, 2))
print("Validation F2:", round(best_score, 3))

Best threshold: 0.32
Validation F2: 0.33


In [ ]:
import pandas as pd

df_full = pd.read_csv("full.csv")

print(df_full.shape)
print(df_full["FLAG"].value_counts())

(42372, 1036)
FLAG
0    38757
1     3615
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# Remove label column
usage = df_full.drop(columns=["FLAG"]).apply(pd.to_numeric, errors="coerce")

# Fill missing values
usage = usage.T.fillna(usage.median(axis=1)).T
usage = usage.fillna(0)

features = pd.DataFrame(index=usage.index)

# Basic usage behaviour
features["mean_usage"] = usage.mean(axis=1)
features["median_usage"] = usage.median(axis=1)
features["max_usage"] = usage.max(axis=1)
features["min_usage"] = usage.min(axis=1)
features["std_usage"] = usage.std(axis=1)

# Usage variation
features["variation"] = (
    features["std_usage"] / (features["mean_usage"] + 1)
)

# Zero / very low usage
features["zero_ratio"] = (usage <= 0.1).mean(axis=1)

# Sudden changes
changes = usage.diff(axis=1)

features["avg_change"] = changes.abs().mean(axis=1)
features["max_drop"] = changes.min(axis=1).abs()
features["max_spike"] = changes.max(axis=1)

# Compare recent 30 days with previous 30 days
recent = usage.iloc[:, -30:].mean(axis=1)
previous = usage.iloc[:, -60:-30].mean(axis=1)

features["recent_change_ratio"] = (
    recent / (previous + 1)
)

# Compare early history with recent history
n = usage.shape[1]

early = usage.iloc[:, :n//4].mean(axis=1)
latest = usage.iloc[:, -n//4:].mean(axis=1)

features["long_term_change"] = (
    latest / (early + 1)
)

# Add theft label
features["FLAG"] = df_full["FLAG"].values

print("WattGuard feature dataset:", features.shape)

features.head()

WattGuard feature dataset: (42372, 13)


,mean_usage,median_usage,max_usage,min_usage,std_usage,variation,zero_ratio,avg_change,max_drop,max_spike,recent_change_ratio,long_term_change,FLAG
0,0.943217,0.000,18.29,0.0,3.094704,1.592567,0.852174,0.236286,10.270,10.040,0.331288,0.067642,1
1,3.939551,0.955,44.57,0.0,6.578796,1.331861,0.048309,0.605537,13.285,43.415,0.678151,7.606892,1
2,7.306889,5.070,79.71,0.0,8.675411,1.044363,0.032850,2.387418,79.710,61.590,0.743610,1.003729,1
3,3.575169,0.000,40.98,0.0,7.943811,1.736288,0.787440,0.620774,20.140,30.590,1.023759,14.286873,1
4,57.656860,52.440,399.40,0.0,38.374234,0.654216,0.040580,4.382244,148.240,351.380,0.199785,1.404953,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Input features + target
X = features.drop(columns=["FLAG"])
y = features["FLAG"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# WattGuard V3 model
model_v3 = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train
model_v3.fit(X_train, y_train)

# Predict
prob = model_v3.predict_proba(X_test)[:, 1]

# Start with threshold 0.40
pred = (prob >= 0.40).astype(int)

# Results
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))

print("\nClassification Report:")
print(classification_report(y_test, pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))

ROC-AUC: 0.764

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.89      7752
           1       0.23      0.55      0.32       723

    accuracy                           0.80      8475
   macro avg       0.59      0.69      0.60      8475
weighted avg       0.89      0.80      0.84      8475


Confusion Matrix:
[[6414 1338]
 [ 328  395]]


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Threshold | Precision | Recall | F1")
print("-----------------------------------")

for t in [0.20, 0.25, 0.30, 0.35, 0.40,
          0.45, 0.50, 0.55, 0.60, 0.65]:

    p = (prob >= t).astype(int)

    precision = precision_score(y_test, p)
    recall = recall_score(y_test, p)
    f1 = f1_score(y_test, p)

    print(
        f"{t:.2f}      | "
        f"{precision:.2f}      | "
        f"{recall:.2f}   | "
        f"{f1:.2f}"
    )

Threshold | Precision | Recall | F1
-----------------------------------
0.20      | 0.13      | 0.86   | 0.22
0.25      | 0.15      | 0.80   | 0.25
0.30      | 0.17      | 0.72   | 0.27
0.35      | 0.19      | 0.63   | 0.29
0.40      | 0.23      | 0.55   | 0.32
0.45      | 0.27      | 0.47   | 0.34
0.50      | 0.30      | 0.40   | 0.34
0.55      | 0.34      | 0.32   | 0.33
0.60      | 0.37      | 0.25   | 0.30
0.65      | 0.42      | 0.20   | 0.27


In [ ]:
# Build reference behaviour using NORMAL consumers only
normal_reference = X_train[y_train == 0]

# Thresholds from normal consumer behaviour
zero_high = normal_reference["zero_ratio"].quantile(0.90)
variation_high = normal_reference["variation"].quantile(0.90)
drop_high = normal_reference["max_drop"].quantile(0.90)
change_high = normal_reference["avg_change"].quantile(0.90)

recent_low = normal_reference["recent_change_ratio"].quantile(0.10)
longterm_low = normal_reference["long_term_change"].quantile(0.10)


def explain_alert(row):

    reasons = []

    if row["zero_ratio"] > zero_high:
        reasons.append("Frequent zero/near-zero readings")

    if row["max_drop"] > drop_high:
        reasons.append("Unusual sudden consumption drop")

    if row["variation"] > variation_high:
        reasons.append("Highly irregular usage pattern")

    if row["avg_change"] > change_high:
        reasons.append("Abnormal day-to-day changes")

    if row["recent_change_ratio"] < recent_low:
        reasons.append("Recent usage dropped sharply")

    if row["long_term_change"] < longterm_low:
        reasons.append("Long-term consumption decline")

    if len(reasons) == 0:
        reasons.append("Combined behaviour differs from normal consumers")

    return " | ".join(reasons[:3])


# Add explanations
results["Why_Flagged"] = results.apply(explain_alert, axis=1)

# Better consumer ID using original dataset index
results["Consumer_ID"] = [
    f"WG-{i:05d}" for i in results.index
]

# Final priority output
final_output = results.sort_values(
    "Risk_Score",
    ascending=False
)

final_output[
    [
        "Consumer_ID",
        "Risk_Score",
        "Risk_Level",
        "Why_Flagged",
        "Inspection",
        "Actual_Label"
    ]
].head(15).reset_index(drop=True)

,Consumer_ID,Risk_Score,Risk_Level,Why_Flagged,Inspection,Actual_Label
0,WG-25868,99.1,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
1,WG-25827,98.2,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
2,WG-26730,95.7,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
3,WG-24177,95.2,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
4,WG-25510,94.9,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,0
5,WG-00206,92.6,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
6,WG-23855,92.5,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
7,WG-00278,91.1,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
8,WG-23063,89.7,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,1
9,WG-39800,89.7,CRITICAL,Unusual sudden consumption drop | Abnormal day...,PRIORITY INSPECTION,0


In [ ]:
import pandas as pd

df = pd.read_csv("../data/wattguard_output.csv")

print(df.columns.tolist())

print(
    df[
        [
            "Consumer_ID",
            "Area",
            "Substation",
            "Feeder_ID"
        ]
    ].head()
)

✅ Chennai demo geography added


,Consumer_ID,Area,Substation,Feeder_ID,Risk_Score,Risk_Level
0,CHN-10001,Ambattur,SS-AMB-07,FDR-01,95.7,CRITICAL
1,CHN-10002,Velachery,SS-VEL-04,FDR-02,92.6,CRITICAL
2,CHN-10003,Mylapore,SS-MYL-11,FDR-03,92.5,CRITICAL
3,CHN-10004,Porur,SS-POR-08,FDR-04,98.2,CRITICAL
4,CHN-10005,Guindy,SS-GDY-05,FDR-05,91.1,CRITICAL
5,CHN-10006,Ambattur,SS-AMB-07,FDR-06,89.7,CRITICAL
6,CHN-10007,Nungambakkam,SS-NGM-10,FDR-07,88.8,CRITICAL
7,CHN-10008,Adyar,SS-ADY-03,FDR-08,88.9,CRITICAL
8,CHN-10009,Ambattur,SS-AMB-07,FDR-09,88.2,CRITICAL
9,CHN-10010,Mylapore,SS-MYL-11,FDR-10,88.5,CRITICAL
